# Exercise: Multilayer Perceptrons

## 1.) Backprop Algo for a Multilayer Perceptron (MLP)

The task is to implement a gradient descent method for a MLP. The structure of the MLP is given below. The algorithm should be implemented using PyTorch API (nn.module and nn.Linear).

## Structure of the Network
<div style="float: left;"><img src="mlp2.png",width=800></div>

**Task 1.1:** Forward Pass and Training


Implement a class TwoLayerNet that performs the foward pass and train the model using the full PyTorch API. 
- Use the PyTorch tutorial and the nn.module, nn.Linear API https://pytorch.org/tutorials/beginner/pytorch_with_examples.html
- Follow the example "PyTorch: Custom nn Modules" but use our data and the network structure given above. 
- Step 1: Start with defining the two layers (hidden layer with 3 units, output layer with 1 unit) in the *\__init\__* function. Use *nn.Linear*! 
- Step 2: After initialization define a member function "*forward(self, x)*" that implements the forward pass of the MLP. Use sigmoid activations. After you have created an object, e.g. model=TwoLayerNet(..), you can call the method by *model(X)*.
- Step 3: Train your model and use *criterion = torch.nn.BCELoss(reduction='sum')* and 
*optimizer = torch.optim.SGD(model.parameters(), lr=alpha)*. Use batches of data to update the network parameters in each iteration.  
- Step 4: Train your model as good as possible (close to label annotation) and plot the loss and the predictions. Change the hyper parameters (number of epochs, learning rate, batch size, number of hidden layers) to achieve this goal (e.g. try batch_size={10,1}, alpha={0.001,0.01,1}, epochs={100,500} and number of hidden units={3, 16}). Evaluate the accuracy of your model on the training and test data (should be around 95%).

In [ ]:
from IPython.display import HTML
from IPython.display import display
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve
import torch
print(torch.__version__)
import torch.nn as nn
import torch.nn.functional as F

import numpy as np

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.axes3d import Axes3D, art3d
from matplotlib.patches import Circle

# stable outputs across different runs
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)


In [ ]:
from sklearn.datasets import make_moons

from torch.utils.data import Dataset, DataLoader


# function to visualize the 2d input data at
def plot_predictions(X,Y,title=None):
    plt.scatter(X[Y[:,0] == 1,0], X[Y[:,0] == 1,1], c='green', label='Positive')
    plt.scatter(X[Y[:,0] == 0,0], X[Y[:,0] == 0,1], c='red', label='Negative')
    plt.legend(loc='lower right')
    plt.title(title)
    plt.xlabel("feature 1")
    plt.ylabel("feature 2")
    plt.show()

class MoonDataset(Dataset):
    def __init__(self, samples,seed):
        X, Y = make_moons(samples, noise=0.23, random_state=seed)
        self.X = torch.from_numpy(X).type(torch.float32)
        self.Y = torch.from_numpy(Y).type(torch.float32).view(-1,1)
        
    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        
        return self.X[idx,:], self.Y[idx]

    
# lets create the moon dataset
dataset=MoonDataset(1000,seed)
print(dataset.X.shape)

# now we split the dataset into a training (80%) and testset (20%)
percent_train = .8
train_size = int((percent_train) * len(dataset))
test_size = len(dataset) - train_size
dataset_train, dataset_test = torch.utils.data.random_split(dataset, [train_size, test_size])


# create DataLoader for batched data
batch_size = 10 #len(dataset_train)
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
batch_size = len(dataset_test)
dataloader_test = DataLoader(dataset_test, batch_size=batch_size,shuffle=True)


# direct access without dataloader
for i in range(5):#range(len(dataloader_train)):
    (x,y) = dataset_train[i]

    print(i, x.shape, y.shape)
    
# access with data loader training data   
for i_batch, (x,y) in enumerate(dataloader_train):
    print(i_batch, x.size(), y.size())
    
# access with data loader test data   
for i_batch, (x,y) in enumerate(dataloader_test):
    print(i_batch, x.size(), y.size())
    
# to plot all data points and classes we have to concatenate all batches
x = torch.FloatTensor()
y = torch.FloatTensor()
for i_batch, (x_batch,y_batch) in enumerate(dataloader_train):
    x = torch.cat((x, x_batch), 0)
    y = torch.cat((y, y_batch), 0)
   
plot_predictions(x,y,title=None)
